# 04 — Failure analysis
Paired B1/E3 predictions, persistent failures and visual inspection. Run from the repository or notebooks directory. No inference or training is performed.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
MET = ROOT / 'outputs/metrics'
OUT = ROOT / 'outputs/figures/failure_analysis'
OUT.mkdir(parents=True, exist_ok=True)
MODELS = ['B0','B1','E1','E2','E3']
STRESSES = ['normal','dark','bright','warm_cast']
frames = []
reference = None
for m in MODELS:
    for s in STRESSES:
        f = MET / f'{m}_seed42_stage2_{s}_test_predictions.csv'
        x = pd.read_csv(f)
        assert len(x)==320 and x.path.is_unique
        assert x.true_label.value_counts().to_dict()=={0:160,1:160}
        assert (x.correct == (x.true_label==x.predicted_label)).all()
        labels = x.set_index('path').true_label.sort_index()
        if reference is None: reference = labels
        else: assert reference.equals(labels)
        frames.append(x.assign(model=m,stress=s))
d = pd.concat(frames,ignore_index=True)
errors = d[d.correct==0].copy()
errors.to_csv(OUT/'all_errors.csv',index=False)
display(pd.crosstab(errors.model,errors.stress).reindex(index=MODELS,columns=STRESSES,fill_value=0))

## Paired comparison
Match the same file and lighting condition. Positive net improvement means fewer errors for E3.

In [ ]:
a=d[d.model=='B1']; b=d[d.model=='E3']
pairs=a.merge(b,on=['path','stress','true_label','fruit','true_class'],suffixes=('_B1','_E3'),validate='one_to_one')
pairs['outcome']=np.select([(pairs.correct_B1==0)&(pairs.correct_E3==1),(pairs.correct_B1==1)&(pairs.correct_E3==0),(pairs.correct_B1==0)&(pairs.correct_E3==0)],['rescued','introduced','both wrong'],default='both correct')
counts=pd.crosstab(pairs.stress,pairs.outcome).reindex(STRESSES)
display(counts)
counts.to_csv(OUT/'paired_counts.csv')
pairs[pairs.outcome!='both correct'].to_csv(OUT/'paired_errors.csv',index=False)
ax=counts[['rescued','introduced','both wrong']].plot.bar(figsize=(9,4),rot=0)
ax.set_ylabel('Image-condition instances'); ax.figure.tight_layout(); ax.figure.savefig(OUT/'paired_errors.png',dpi=300); plt.show()

## Persistent errors and fruit types
Repeated errors are not independent images.

In [ ]:
persistent=errors.groupby(['path','true_class']).size().sort_values(ascending=False).rename('error_instances')
display(persistent.head(15)); persistent.to_csv(OUT/'persistent_errors.csv')
by_fruit=pd.crosstab(errors[errors.stress!='normal'].fruit,errors[errors.stress!='normal'].model).reindex(columns=MODELS,fill_value=0)
display(by_fruit); by_fruit.to_csv(OUT/'stressed_errors_by_fruit.csv')

## Four representative examples
Two corrected rotten cases, one newly missed rotten case, one persistent failure. Selected after evaluation for explanation, not tuning.

In [ ]:
choices=[('RottenStrawberry (182).jpg','warm_cast'),('RottenJujube (196).jpg','warm_cast'),('RottenGrape (179).jpg','bright'),('RottenGrape (183).jpg','warm_cast')]
selected=pd.concat([pairs[(pairs.path.str.endswith('/'+name))&(pairs.stress==s)] for name,s in choices],ignore_index=True)
assert len(selected)==len(choices)
display(selected[['path','stress','true_class','outcome','probability_rotten_B1','probability_rotten_E3']])
selected.to_csv(OUT/'selected_examples.csv',index=False)

In [ ]:
from PIL import Image
from src.preprocess.stress import apply_stress
from src.preprocess.pipeline import PIPELINES
fig, axes=plt.subplots(len(selected),4,figsize=(13,3.6*len(selected)),squeeze=False)
for i,r in selected.iterrows():
    path=ROOT/r.path
    if not path.is_file():
        for ax in axes[i]: ax.axis('off')
        axes[i,0].text(.5,.5,'Image unavailable',ha='center')
        continue
    with Image.open(path) as im: rgb=np.array(im.convert('RGB').resize((250,250),Image.Resampling.BILINEAR))
    observed=apply_stress(rgb,r.stress)
    images=[rgb,observed,PIPELINES['B1'](observed.copy()),PIPELINES['E3'](observed.copy())]
    titles=['Original: '+r.true_class,r.stress,f"B1: {r.predicted_class_B1} (P(rot)={r.probability_rotten_B1:.1%})",f"E3: {r.predicted_class_E3} (P(rot)={r.probability_rotten_E3:.1%})"]
    for ax,img,title in zip(axes[i],images,titles): ax.imshow(img); ax.set_title(title,fontsize=10); ax.axis('off')
    axes[i,0].set_title(Path(r.path).name+'\n'+titles[0],fontsize=9)
fig.tight_layout(); fig.savefig(OUT/'failure_examples.png',dpi=300,bbox_inches='tight'); plt.show()

## Interpretation
Inspect actual images before proposing causes. Compare colour, texture, clipping, shadows and visible decay. These are hypotheses, not established causal mechanisms. Report persistent failures and examples where E3 hurts as well as helps.